## MLP Network Review - Digit Recognition

다음은 784개 neuron의 input layer와 32개 neuron의 hidden layer 2개, 그리고 10개의 neuron을 가진 output layer로 구성된 Fully Connected 신경망(Neural Network)이다.

<img src="https://drive.google.com/uc?export=view&id=1dFSEq0iYzucwZ9E3ERr-JnasI4dVQim0" width="700" />

In [ ]:
from keras.datasets import mnist
from keras.models import Sequential

from keras.layers import Dense, Activation
from keras.utils import to_categorical
#from keras.utils import np_utils

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(60000, 784)
X_test = X_test.reshape(10000, 784)
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# normalize the feature data
X_train /= 255
X_test /= 255

# one hot encoding the label
nb_classes = 10
Y_train = to_categorical(y_train, nb_classes)
Y_test = to_categorical(y_test, nb_classes)

In [ ]:
mlp = Sequential([
    Dense(32, input_shape=(784,)),
    Activation('relu'),
    Dense(32),
    Activation('relu'),
    Dense(10),
    Activation('softmax')
])

mlp.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
mlp.fit(X_train, Y_train,
          batch_size=128, epochs=20,
          verbose=1)

In [ ]:
score = mlp.evaluate(X_test, Y_test)
print('Test score:', score[0])
print('Test accuracy:', score[1])

# Convolutional Neural Network


* 일반 DNN에서 2차원의 digit 이미지는 1차원으로 변환되어 학습이 된다. 따라서 이미지의 지역적/공간적 정보를 잃게 된다.


<img src="https://miro.medium.com/max/700/1*CnNorCR4Zdq7pVchdsRGyw.png" width="1200"/><center>*CNN for Digit Recognition*</center>


## Convolution

<img src="https://t1.daumcdn.net/cfile/tistory/9943EB425AF9A5272B?download" width="1200"/>

***Convolution 참고 자료***
* [Different Kinds of Convolutional Filters](https://www.saama.com/different-kinds-convolutional-filters/)

* [Kernel(image procesing) @ Wikipedia](https://en.wikipedia.org/wiki/Kernel_(image_processing))

* [Convolutions with OpenCV and Python](https://www.pyimagesearch.com/2016/07/25/convolutions-with-opencv-and-python/)

* [Convolution의 정의](https://talkingaboutme.tistory.com/entry/DL-Convolution의-정의)



In [ ]:
import numpy as np
from scipy.ndimage.filters import convolve
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

k1 = np.array([[-1, 0, 1],
               [-1, 0, 1],
               [-1, 0, 1] ])
k2 = np.array([[1, 1, 1],
               [0, 0, 0],
               [-1, -1, -1] ])
k3 = np.array([[-1, -1, -1],
               [-1, 8, -1],
               [-1, -1, -1] ])
k4 = np.array([[0, 0, 1],
               [0, 0, 0],
               [-1, 0, 0] ])
k5 = np.array([[0, 1, 0],
               [1, 0, 1],
               [0, 1, 0] ])
k6 = np.array([[0, 0, 0],
               [0, 0, 0],
               [0, 0, 0] ])

img1 = Image.open("/content/digit.png")
img1 = img1.convert('LA')

np_frame = np.array(img1.getdata())
np_img = np_frame[:,1].reshape(280,280)


In [ ]:

res = convolve(np_img, k4)

plt.subplot(1,2,1)
plt.imshow(np_img, cmap='gray')
plt.subplot(1,2,2)
plt.imshow(res, cmap='gray')
plt.show()

## Pooling(Sub Sampling) and Max Pooling
* Fully Connected Network은 Train 데이터에 과적합(Overfit)이 생길수 있다.
* 필요없는 연산도 수행되어 학습 효율이 떨어진다

<img src="https://miro.medium.com/max/652/1*Sh9e6Hzx8ZcOinuLvy8Fmw.png" width="1200"/>


In [ ]:
import skimage.measure

pooled = skimage.measure.block_reduce(np_img, (2,2), np.max)
pooled = skimage.measure.block_reduce(pooled, (2,2), np.max)
pooled = skimage.measure.block_reduce(pooled, (2,2), np.max)


In [ ]:
plt.subplot(1,2,1)
plt.imshow(np_img, cmap='gray')
plt.subplot(1,2,2)
plt.imshow(pooled, cmap='gray')
plt.show()

## CNN을 이용한 Digit Recognition

In [ ]:
import numpy as np
from keras.datasets import mnist     # MNIST dataset is included in Keras
(X_train, y_train), (X_test, y_test) = mnist.load_data()


In [ ]:
# Let's backup the test image data for later use
import copy
X_test_copy = copy.deepcopy(X_test) # backup the test data

In [ ]:
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)
input_shape = (28, 28, 1)

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

<img src="https://miro.medium.com/max/700/1*CnNorCR4Zdq7pVchdsRGyw.png" width="1200"/><center>*CNN for Digit Recognition*</center>



In [ ]:
from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation, Dropout, Flatten
from keras.layers import MaxPooling2D, Conv2D

model = Sequential()
model.add(Conv2D(32, kernel_size=(3,3), input_shape=input_shape, padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, kernel_size=(3,3), padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))

model.add(Dropout(0.2))
model.add(Dense(10))
model.add(Activation('softmax'))

model.summary()

In [ ]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(x=X_train,y=y_train, epochs=10)

In [ ]:
score = model.evaluate(X_test, y_test)
print('Test score:', score[0])
print('Test accuracy:', score[1])

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['figure.figsize'] = (9,9) # Make the figures a bit bigger
plt.subplot()
idx = 95
plt.imshow(X_test_copy[idx], cmap='gray', interpolation='none')

In [ ]:
print(np.argmax((model.predict(X_test[idx:idx+1]))[0].reshape(10,-1)))
(model.predict(X_test[idx:idx+1]))[0].reshape(10,-1)

# MLP 과 CNN 의 예측 비교

### Handwriting Web


In [ ]:
from IPython.display import HTML
from google.colab.output import eval_js
from base64 import b64decode
from io import BytesIO
from PIL import Image

canvas_html = """
<canvas width=%d height=%d></canvas>
<button>Finish</button>
<script>
var canvas = document.querySelector('canvas')
var ctx = canvas.getContext('2d')
ctx.lineWidth = %d
var button = document.querySelector('button')
var mouse = {x: 0, y: 0}

canvas.addEventListener('mousemove', function(e) {
  mouse.x = e.pageX - this.offsetLeft
  mouse.y = e.pageY - this.offsetTop
})
canvas.onmousedown = ()=>{
  ctx.beginPath()
  ctx.moveTo(mouse.x, mouse.y)
  canvas.addEventListener('mousemove', onPaint)
}
canvas.onmouseup = ()=>{
  canvas.removeEventListener('mousemove', onPaint)
}
var onPaint = ()=>{
  ctx.lineTo(mouse.x, mouse.y)
  ctx.stroke()
}

var data = new Promise(resolve=>{
  button.onclick = ()=>{
    resolve(canvas.toDataURL('image/png'))
  }
})
</script>
"""

def convertImage(image):
    img = image.resize((28,28))
    img = img.convert('LA')
    np_frame = np.array(img.getdata())
    digit = np_frame[:,1].reshape(28,28)
    digit = digit / 255
    return image, digit

def draw(filename='digit.png', w=280, h=280, line_width=20):
    display(HTML(canvas_html % (w, h, line_width)))
    data = eval_js("data")
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    image = Image.open(BytesIO(binary))
    return convertImage(image)


def loadImage(f='digit.png'):
    image = Image.open(f)
    return convertImage(image)

### 샘플 숫자 추출

In [ ]:
image, d2 = draw()

In [ ]:
image, d2 = loadImage('digit.png')

### 비교

In [ ]:
display(image)

MLP 의 예측

In [ ]:
np.argmax(mlp.predict([d2.reshape(-1,784)])[0])


CNN의 예측

In [ ]:
data = d2.reshape(1,28,28,1)
np.argmax(model.predict(data)[0])

### saving the ML model

In [ ]:
mlp.save('/content/drive/MyDrive/mlp')

In [ ]:
model.save('/content/drive/MyDrive/mlp')